# 21.16 — MP-04. Согласованная 4D-геометрия

**Статус: перенос одной опорной поверхности прямыми полями Greedy на две фазы; полная согласованная геометрия 15 фаз не построена.** Это технический pilot на Георге, основанный на [MP-03](21.15_MP03_Регистрация_фаз_4D_КТ.ipynb). Соответствие вершин показано для `phase_00 → phase_01` и `phase_00 → phase_07`; анатомическая истинность не доказана.

Связи: [код парного расчёта](mp03_04_monai_pair.py), [отчёт](mp03_04_report.md), [QC 01](mp03_04_georg_00_01_monai_qc.json), [QC 07](mp03_04_georg_00_07_monai_qc.json), [методологический план](README.md).

## Основной MP-04 на Нике: одна поверхность, исходная сетка

[Основной отчёт регистрации и FEM-допуска](mp03_nix_fullres_report.md) использует реальную 4D-КТ Ника. Одна поверхность фазы 00 (`152 637` вершин, `305 114` треугольников) перенесена в 01 и 07 с неизменной связностью. Средняя дистанция вершин до отдельных автоматических поверхностей уменьшилась `1,522→0,997` и `1,516→0,875` мм. Для 07 взято только сглаженное поле с положительным sampled Jacobian; исходное поле с 115 складками отклонено.

Это техническое согласование автоматических продуктов. Все три маски касаются верхней границы КТ, а принятой ручной разметки целевых фаз нет. Поэтому форму и движение нельзя объявить анатомически точными. Старые разделы о Георге ниже остаются историческим пилотом. [Read-only FEM-preflight](mp02_nix_fem_preflight.json) блокирует подмену FEM-геометрии до независимой проверки.


## Дополнение 30.09.2026: согласованная поверхность во всех фазах Ника

Одна опорная автоматическая поверхность (`152 637` вершин, `305 114` треугольников) перенесена полями MP-03 на все 11 целевых фаз с сохранением связности. Средняя дистанция тех же вершин до отдельной автоматической целевой поверхности уменьшилась во всех парах, в среднем на `0,638` мм. На фазе 03 остаточный 95-й процентиль дистанции равен `7,016` мм; это значимое ограничение геометрического совпадения. Подробности — в [отчёте](mp03_nix_full_cycle_report.md) и [агрегате](mp03_nix_full_cycle.json).

Согласованная нумерация вершин полезна для расчётного конвейера, но не устанавливает анатомическую правильность деформации. [Пофазная проверка FOV](mp03_nix_fov_contact_report.md) установила контакт наружной маски `whole_heart` с верхом КТ во всех фазах, но маски крови и миокарда туда не доходят. Независимых ручных фазовых контуров нет. Передача этих поверхностей в FEM и сравнение импеданса остаются заблокированы до анатомического допуска.

## Входы и способ переноса

Опорная маска `whole_heart` фазы 00 и отдельные целевые маски фаз 01/07 получены TotalSegmentator v2. CT — измеренные реконструкции, маски — результаты автоматического алгоритма; независимого анатомического референса нет. Подтверждение RAS/мм дано conversion manifest, поскольку NIfTI-заголовок содержит `unknown units`. Фазам не назначены ED/ES, дыхательное состояние и принятый общий R–R цикл.

На общей уменьшенной сетке `1.377×1.377×1.25` мм из одной опорной маски marching cubes построена **одна поверхность: 51 259 вершин, 102 522 треугольника**. Те же вершины и грани служат неподвижным, жёстким и Greedy-кандидатом. Greedy переносит вершины `moving_to_fixed` в физических координатах ITK LPS; маску переносит противоположный `fixed_to_moving` как pullback с nearest-neighbor interpolation. Это различие направлений необходимо для корректного сравнения. Никакие новые auto masks не выдаются за независимую истину.

## Геометрия и поверхности на общей сетке

| Цель | Средняя дистанция одних и тех же вершин до целевой auto-поверхности, мм: неподвижная / жёсткая / Greedy | P95 Greedy, мм | Вершины Greedy вне crop | Разница объёма маски, мл: неподвижная → Greedy | Разница площади по граням, мм²: неподвижная → Greedy |
|---|---:|---:|---:|---:|---:|
| 01 | 1.216 / 1.223 / **0.920** | 1.946 | 0 | +17.773 → **+2.725** | +741 → +3706 |
| 07 | 1.462 / 1.487 / **0.991** | 2.442 | 0 | +29.198 → **+2.266** | +1833 → +7038 |

**Результат модели:** у двух перенесённых вариантов сохраняется нумерация и связность одной опорной поверхности. Средняя дистанция до отдельных auto-поверхностей уменьшилась при Greedy. Объёмное расхождение тоже уменьшилось, но дискретная площадь по открытым граням вокселей **ухудшилась** в обоих случаях. Эта площадь зависит от сетки, staircasing и интерполяции и не равна гладкой анатомической площади. Результаты не разделяют истинное движение, ошибку сегментации и ошибку представления.

## Поле и QC геометрии

Сохранённые **локально** поля `runs/mp03_04_greedy_field_00_01.npz` и `runs/mp03_04_greedy_field_00_07.npz` являются pullback цель → источник в LPS, мм. Поля имеют физическую сетку и контрольные суммы в [парных](mp03_04_georg_00_01_monai_qc.json) [JSON](mp03_04_georg_00_07_monai_qc.json); сами поля не включены в Git. На уменьшенной сетке минимум `det J` равен 0.419 и 0.347, неположительных значений не найдено. Максимальная ошибка композиции прямого/обратного преобразований на контрольных точках 0.069 и 0.089 мм. Pullback выходит за moving crop в 0.223% и 0.354% отсчётов. Вершины поверхности за crop не вышли. Исходные маски внутри полного FOV, crop касается верхней границы.

**Ограничение:** положительность дискретного Якобиана не гарантирует отсутствие складок между узлами или на исходном разрешении. Контроль касается двух пар; единой принятой геометрии всех 15 фаз ещё нет. Первая пара при повторе дала близкие, но не одинаковые поля. Вклад ошибок регистрации и auto-сегментации неидентифицируем без независимой проверки.

In [ ]:
# Сверка сохранённых агрегатов и постоянной связности.
import json
from pathlib import Path

for target in ("01", "07"):
    d = json.loads(Path(f"mp03_04_georg_00_{target}_monai_qc.json").read_text(encoding="utf-8"))
    s = d["same_surface_qc"]
    assert (s["vertex_count"], s["triangle_count"]) == (51259, 102522)
    assert s["same_vertices_and_faces_for_all_three_candidates"]
    assert s["greedy_vertices_outside_crop_fraction"] == 0
    print(target, "mean vertex distance mm",
          round(s["mean_distance_mm"]["stationary"], 3), "->",
          round(s["mean_distance_mm"]["greedy"], 3),
          "area difference mm2",
          round(d["mask_qc"]["greedy"]["voxel_face_area_difference_mm2"]))

## Решение и граница применения

Техническое требование одинаковых вершин выполнено для двух пар на общей уменьшенной сетке. Согласование с отдельными автоматическими масками по Dice/ASSD/расстоянию вершин стало лучше, но площадь по дискретным граням и ошибка центра для 07 — хуже. **Статус: кандидатная двухпарная геометрия, не валидированная 4D-геометрия сердца.**

На исходном разрешении QC выполнен для двух пар; для остальных фаз ещё нужны такие же проверки, пространственный контроль у краёв, временная привязка, повторяемость и независимое анатомическое сравнение, если доступно. Влияние на FEM-импеданс и УО/ФВ не измерено. КТ, маски, поверхность и поле остаются локально.

## Исходная сетка и граница передачи в FEM

На исходном шаге КТ одна поверхность фазы 00 (`359 829` вершин, `719 662` треугольника) перенесена с неизменной связностью. Средняя дистанция её вершин до целевой автоматической поверхности уменьшилась `0,855→0,593` мм для 01 и `1,084→0,760` мм для 07 с усиленным сглаживанием. Положительный det J на узлах полного crop для этих двух полей — технический критерий. Независимая анатомическая точность, качество поверхности между узлами и привязка к FEM-координатам не доказаны. Площадь ступенчатой перенесённой маски, измеренная по воксельным граням, ухудшилась.

[Подробный полноразрешённый QC](mp03_fullres_report.md) и [FEM preflight Георга](mp02_fem_transfer_preflight.md) фиксируют решение: **ни одна зарегистрированная геометрия ещё не допущена в индивидуальную FEM-постановку**. Отсутствуют принятая независимая разметка этих фаз и проверенная карта Georg 4D-КТ→Georg FEM. Поэтому новые `Z`/`ΔZ` не рассчитывались. После допуска сравнение следует проводить на одной и той же Georg-сетке с одинаковыми CEM-контактами, тканевыми свойствами и фоном, меняя только сердечную геометрию.
